# 01 文档切块与向量化

**目标**：把非结构化文档变成可检索的向量。这是 RAG 的第一步，也是决定效果上限的一步。

三个关键决策：
1. **切多大**——太小丢上下文，太大检索不准。本课按小节切，保证语义完整
2. **怎么向量化**——本例用 TF-IDF 字符 n-gram，不依赖任何嵌入模型，断网可跑
3. **怎么算相似**——L2 归一化后点积即余弦相似度

In [1]:
import sys, os
sys.path.insert(0, '../..')

from src.rag_core import load_documents, chunk_documents

docs = load_documents('../../data/kb')
print(f'载入 {len(docs)} 篇文档')
for d in docs:
    print(f"  {d['source']:24s} {len(d['text']):5d} 字")

载入 3 篇文档
  PyTorch常见错误.md            1487 字
  模型部署指南.md                 1449 字
  深度学习基础.md                 1050 字


## 1. 切块

按二级标题切，标题保留在块内，这样检索时能带上上下文。

In [2]:
chunks = chunk_documents(docs)
print(f'切成 {len(chunks)} 个片段\n')
for c in chunks[:5]:
    print(f"[{c['source']}] {c['title']}  ({len(c['text'])} 字)")

切成 16 个片段

[PyTorch常见错误.md] 张量形状不匹配  (332 字)
[PyTorch常见错误.md] 设备不一致  (347 字)
[PyTorch常见错误.md] 梯度爆炸与梯度消失  (304 字)
[PyTorch常见错误.md] 显存不足  (262 字)
[PyTorch常见错误.md] 参数更新没有生效  (214 字)


## 2. 向量化

中文没有空格，`TfidfVectorizer` 默认的按词切分会失效（整句被当成一个词）。
改用 `analyzer='char_wb'` 走字符 n-gram，这是中文场景最简单有效的办法。

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

texts = [c['text'] for c in chunks]
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 3),
                      max_features=20000, sublinear_tf=True)
matrix = normalize(vec.fit_transform(texts))

print('向量矩阵形状:', matrix.shape)
print('每个片段是一个', matrix.shape[1], '维向量')

向量矩阵形状: (16, 5071)
每个片段是一个 5071 维向量


## 3. 检索

归一化之后，两个向量的点积就是余弦相似度。

In [4]:
import numpy as np

def search(query, k=3):
    q = normalize(vec.transform([query]))
    scores = np.asarray((matrix @ q.T).todense()).ravel()
    for i in np.argsort(-scores)[:k]:
        print(f"  {scores[i]:.3f}  {chunks[i]['source']} · {chunks[i]['title']}")

for q in ['矩阵维度不匹配怎么排查', '怎么防止过拟合', '上线后怎么回滚']:
    print(f'问题：{q}')
    search(q)
    print()

问题：矩阵维度不匹配怎么排查
  0.195  PyTorch常见错误.md · 张量形状不匹配
  0.019  模型部署指南.md · 服务的可观测性
  0.000  PyTorch常见错误.md · 梯度爆炸与梯度消失

问题：怎么防止过拟合
  0.162  深度学习基础.md · 过拟合与正则化
  0.078  深度学习基础.md · 神经网络的基本结构
  0.017  深度学习基础.md · 激活函数的作用

问题：上线后怎么回滚
  0.094  模型部署指南.md · 服务的可观测性
  0.079  模型部署指南.md · 灰度发布与回滚
  0.024  模型部署指南.md · 常见上线事故



## 小结

- 切块策略直接决定检索质量：切碎了丢上下文，切大了噪声多
- TF-IDF 是「词面匹配」，好处是完全离线、可解释；缺点是同义词检索不到
- 生产环境换成 `bge-m3`、`text-embedding-3` 这类语义嵌入，**检索接口不变**，只是把 `vec.transform` 换成调嵌入 API
- 下一步：把检索结果拼进 prompt，交给大模型生成答案